In [2]:
import os
from pathlib import Path

root_path = Path.cwd().parent 
os.chdir(root_path)

print(f"Current working directory: {os.getcwd()}")

import json
import sqlite3
from app.config import get_settings
from eval.llm import get_response

Current working directory: /Users/ron/Documents/google-drive-RAG


### chunks db client fucntion

In [11]:
conn = sqlite3.connect("eval/data/test_chunks.db")
def get_all_chunks():
    query = "SELECT chunk_id, chunk_text FROM chunks"
    cursor = conn.execute(query)
    result = [{"chunk_id": chunk_id, "chunk_text": chunk_text} for chunk_id, chunk_text in cursor.fetchall()]
    return result

def get_chunks_by_ids(chunk_ids: list[str]):
    if not chunk_ids:
        return []

    placeholders = ", ".join("?" for _ in chunk_ids)
    query = f"""SELECT chunk_id, chunk_text 
    FROM chunks 
    WHERE chunk_id IN ({placeholders})"""
    cursor = conn.execute(query, chunk_ids)
    result = [{"chunk_id": chunk_id, "chunk_text":chunk_text} for chunk_id, chunk_text in cursor.fetchall()]

    return result

### Retrieval Pipeline run on testing db

In [23]:
## ====================keyword search========================
import bm25s 

class BM25Retriever:
    def __init__(self):
        self.chunks_with_ids = get_all_chunks()
        self.chunk_texts = [chunk['chunk_text'] for chunk in self.chunks_with_ids]
        self.chunk_ids = [chunk['chunk_id'] for chunk in self.chunks_with_ids]

        self.tokenized_corpus = bm25s.tokenize(self.chunk_texts)
        self.bm25_retriever = bm25s.BM25()
        self.bm25_retriever.index(self.tokenized_corpus)

    def retrieve(self, query: str, k: int = 50) -> tuple[list[str], list[float]]:
        query_tokens = bm25s.tokenize([query])
        results, scores = self.bm25_retriever.retrieve(query_tokens, k=k)
        return {"ids": self.map_results_to_chunks_ids(results[0]), "scores": scores.tolist()}
    
    def map_results_to_chunks_ids(self, results: list[int]) -> list[str]:
        return [self.chunk_ids[i] for i in results]
    

# ===========================semantic search========================
import app.embedding as embedding
from eval.utils.chroma import Chroma
from app.config import Settings

def get_top_k_chunks(query, settings : Settings, k=50):
    query_embedding = embedding.embed_text(query, settings)
    chroma_client = Chroma()

    return chroma_client.retrieve_similar_documents(query_embedding, n_results=k)


#==========================hybrid search========================
from app.reranking import Reranker

reranker = Reranker()

def rrf(rankings: list[list[str]], k: int = 60) -> list[tuple[str, float]]:
    
    ## reciprocal_rank_fusion
    ## Performs Reciprocal Rank Fusion on multiple ranked lists.
    
    rrf_scores = {}
    
    for ranked_list in rankings:
        # rank starts at 1 for the formula
        for rank, doc_id in enumerate(ranked_list, start=1):
            if doc_id not in rrf_scores:
                rrf_scores[doc_id] = 0.0
            
            # Apply the standard RRF formula
            rrf_scores[doc_id] += 1.0 / (k + rank)
            
    # Sort documents by their calculated RRF score in descending order
    sorted_results = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    return sorted_results


def hybrid_search(query: str, settings : Settings, k:int =20):
    keyword_results = BM25Retriever().retrieve(query)
    semantic_results = get_top_k_chunks(query, settings=settings)
    #rrf ranking
    rrf_results = rrf([keyword_results['ids'], semantic_results['ids'][0]])
    # get all chunks text
    chunks = get_chunks_by_ids(chunk_ids=[chunk_id for chunk_id, _ in rrf_results])
   
    chunk_texts = [chunk['chunk_text'] for chunk in chunks]
    map_chunk_id_text = {
        chunk['chunk_text']: chunk['chunk_id']
        for chunk in chunks
        } 
    
    #rerank
    reranked_results = reranker.rerank(query, chunk_texts)
    
    result = []
    for text, _ in reranked_results[:k]:
        chunk_id = map_chunk_id_text[text]
        result.append(
            {"chunk_text": text, 
             "chunk_id": chunk_id, 
            }
            )
    
    return result



Loading weights: 100%|██████████| 393/393 [00:00<00:00, 20832.37it/s]


### prompt for generation eval (LLM as a judge)

In [30]:
judge_prompt = """
Question:
{question}

Retrieved Context:
{context}

Ground Truth:
{ground_truth}

Model Answer:
{prediction}

Evaluate (score only using binary 0/1):

1. Correctness (0 or 1)
2. Groundedness (0 or 1)

Groundedness means every factual claim is supported by the context.

Return JSON:

{{
  "correctness": int,
  "groundedness": int,
  "reason": string
}}
"""

### Evaluation loop

In [33]:
settings = get_settings()
# load test data
with open("eval/test_data_labeled.json", "r", encoding="utf-8") as file:
    test_data = json.load(file)

# metrics
hit_3 = 0
mrr = 0
correctness = 0
groundedness = 0
judge_results = []
for i, row in enumerate(test_data):
    print("========================")
    print(f"data {i+1}:")

    question = row["question"]
    ground_truth = row["ground_truth"]
    relavant_chunk_id = row["reference_chunk_id"]

    print("Question: " ,question)
    retrieved_chunks = hybrid_search(query=question, settings=settings, k=3)
    
    chunks_context = "Context:\n" + "\n".join(chunk['chunk_text'] for chunk in retrieved_chunks)

    # get llm answer
    prediction = get_response(prompt = chunks_context+"\n"+question)

    print("Model answer: ", prediction)
    # llm as a judge to evaluate the model answer
    prompt = judge_prompt.format(
        question=question, 
        ground_truth=ground_truth, 
        context=chunks_context, 
        prediction=prediction
        )
    judge_result = get_response(prompt=prompt)
    judge_results.append(judge_result)
    # compute the llm judge metric score
    correctness += judge_result['correctness']
    groundedness += judge_result['groundedness']

    # compute the retrieval performance metric
    if relavant_chunk_id in [chunk['chunk_id'] for chunk in retrieved_chunks]:
        hit_3 += 1
        print("hit@3: 1")
        if relavant_chunk_id == retrieved_chunks[0]['chunk_id']:
            mrr += 1
            print("MRR: 1")
        elif relavant_chunk_id == retrieved_chunks[1]['chunk_id']:
            mrr += (2/3)
            print("MRR: 0.66")
        elif relavant_chunk_id == retrieved_chunks[2]['chunk_id']:
            mrr += (1/3)
            print("MRR: 0.33")
    else:
        print("hit@3: 0")
        print("MRR: 0")
    
hit_3 /= len(test_data)
mrr /= len(test_data)
correctness /= len(test_data)
groundedness /= len(test_data)

data 1:
Question:  What is the Basic Pay amount listed in the EARNINGS section of the payslip?


Model answer:  {'answer': '1,800.00'}
hit@3: 1
MRR: 0.66
data 2:
Question:  What is the Basic Pay amount listed in the Earnings section of Peng Rong Tan's final payslip for May 2026?


Model answer:  {'basic_pay_amount': '1,800.00'}
hit@3: 1
MRR: 1
data 3:
Question:  What is the basic pay amount listed in the earnings section of the payslip for Peng Rong Tan?


Model answer:  {'basic_pay': 1800.0}
hit@3: 1
MRR: 1
data 4:
Question:  According to the assumptions, which group of people is seated in the area?


Model answer:  {'answer': 'guests'}
hit@3: 1
MRR: 1
data 5:
Question:  What is the name of the person who received the AWS Academy Graduate - Cloud Architecting - Training Badge?


...

Model answer:  {'section_name': 'Chapter 3: Business Opportunities', 'explanation': 'The instructions to describe business opportunities and entrepreneurial skills are located in Section 3.4 of the final report template.'}
hit@3: 1
MRR: 1
data 97:
Question:  What section of the document is designated for stating opinions on industry experiences, future expectations, and recommendations regarding the Industrial Training scheme?


Model answer:  {'document': 'final_report_template.docx', 'section': 'Chapter 4: Conclusions & Recommendations'}
hit@3: 1
MRR: 1
data 98:
Question:  What specific task did the individual execute regarding laptop software according to the IT Support section?


Model answer:  {'error': 'Information insufficient', 'explanation': 'The provided text sections (1.1, 1.2, and 1.5) do not contain any information regarding IT Support tasks or specific actions related to laptop software.'}
hit@3: 0
MRR: 0
data 99:
Question:  What was the CGPA achieved in the Diploma in Information Technology?


Model answer:  {'error': 'The provided document contains academic transcripts for a Bachelor in Data Science (Honours), not a Diploma in Information Technology. Therefore, the CGPA for a Diploma in Information Technology cannot be determined from this text.'}
hit@3: 0
MRR: 0
data 100:
Question:  What is the CGPA listed for the Bachelor in Data Science of Computer Science degree?


Model answer:  {'answer': '3.8'}
hit@3: 1
MRR: 0.33


### Result Metric

In [35]:
print("hit@3: ", hit_3)
print("MRR@3: ", mrr)
print("correctness@3: ", correctness)
print("groundedness@3: ", groundedness)

hit@3:  0.75
MRR@3:  0.6833333333333332
correctness@3:  0.82
groundedness@3:  0.91


*Hit@3: 0.75* :<br>     75% of the time, the correct chunk is listed on top 3 out of 100. <br><br>
*MRR@3:  0.68* :<br>    When the correct chunk appears on top 3, it is usually ranked #1 or #2.<br><br>
*Correctness@3:     0.82* :<br> By retrieve only 3 chunks. 82% of the time, model answers user's question correctly.<br><br>
*Groundedness@3:    0.91* :<br> The model is almost always sticking to retrieved chunks.

In [ ]:
for i, result in enumerate(judge_results):
    print(f"{i+1}: \ncorrectness: {result["correctness"]}\ngroundedness: {result["groundedness"]}\nReason:{result["reason"]}")
    print("=================================")

[{'correctness': 1,
  'groundedness': 1,
  'reason': "The model answer '1,800.00' matches the Ground Truth exactly. The value is explicitly listed under the EARNINGS section as 'Basic Pay' in all provided payslip contexts."},
 {'correctness': 1,
  'groundedness': 1,
  'reason': 'The model answer correctly identifies the Basic Pay amount as 1,800.00 from the May 2026 payslip context, matching the ground truth exactly.'},
 {'correctness': 1,
  'groundedness': 1,
  'reason': "The model answer correctly identifies the basic pay amount as 1800.0, which matches the ground truth and is explicitly stated as '1,800.00' for Basic Pay in all provided payslip contexts for Peng Rong Tan."},
 {'correctness': 1,
  'groundedness': 1,
  'reason': "The model answer 'guests' matches the ground truth exactly. Additionally, the retrieved context explicitly states in Assumption 1 that 'only guests are sited', confirming that the answer is fully grounded in the provided text."},
 {'correctness': 1,
  'ground